First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings. 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select, valid_test_select_per_user
from initialize_model import initialize_mu_b_c
from helpers import get_rmse, get_ndcg
from fit_model import fit_model_no_UV, fit_model_full

In [3]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [4]:
df = load_data()

In [5]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [6]:
df = df[df.rating > 0]

In [7]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [8]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [9]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())

In [10]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [11]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [12]:
# Filtering users and books with enough observations

min_rats = 5
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

In [13]:
print("Y shape:", Y.shape)
print("total entries:", (Y > 0).sum().sum())
print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))
print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))

Y shape: (6916, 8903)
total entries: 113019
avg number of ratings per user: 16.34
avg number of ratings per book: 12.69


In [14]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

# Selecting validation and test sets

In [15]:
# Y_train, valid_data, test_data = valid_test_select(Y, 20_000, 20_000, seed=102)
Y_train, valid_data, test_data = valid_test_select_per_user(Y, seed=102)

In [16]:
print("Y_train shape:", Y_train.shape)  # (14903, 22661)
print("total train entries:", (Y_train > 0).sum().sum())  # 142612
print("avg. # train entries/user:", f"{(Y_train > 0).sum(axis=1).mean():.2f}")  # 9.57
print("avg. # train entries/book:", f"{(Y_train > 0).sum(axis=0).mean():.2f}")  # 6.29
print("number of validation samples:", valid_data.shape[0])
print("number of test samples:", test_data.shape[0])

Y_train shape: (6916, 8903)
total train entries: 83288
avg. # train entries/user: 12.04
avg. # train entries/book: 9.36
number of validation samples: 20407
number of test samples: 9324


# ALS

In [17]:
Y_csr = csr_matrix(Y_train)
Y_csc = Y_csr.tocsc()

## Model with just a training global mean

In [18]:
mu, _, _ = initialize_mu_b_c(Y_train)
print("test RMSE of model=mu:", get_rmse(mu, test_data))
preds_ndcg = np.full(Y_train.shape, mu)
print("test NDCG of model=mu:", get_ndcg(preds_ndcg, Y_train, test_data, 5))
# test rmse of model=mu: 1.8087624016357353

test RMSE of model=mu: 1.7901900251548002
test NDCG of model=mu: 0.0


## Model with only biases

In [19]:
mu, b, c = initialize_mu_b_c(Y_train)
mu_bias, b_bias, c_bias = fit_model_no_UV(
    Y_train, Y_csr, Y_csc, mu, b, c, 3.6, valid_data, 1e-5, info=1
)

results: counter =  22, max_norm_diff =    0.04, valid_rmse = 1.600406, valid_ndcg = 0.000526, valid_rmse_diff = 8.6e-06, valid_ndcg_diff = 0      


In [20]:
test_preds_rmse = np.clip(
    mu_bias + b_bias[test_data.rows] + c_bias[test_data.cols], 1, 10
)
test_preds_ndcg = mu_bias + b_bias[:, None] + c_bias[None, :]
test_rmse_biases = get_rmse(test_preds_rmse, test_data)
test_ndcg_biases = get_ndcg(test_preds_ndcg, Y_train, test_data, 5)
print("test RMSE of a tuned bias-only model:", test_rmse_biases)
print("test NDCG of a tuned bias-only model:", test_ndcg_biases)
# min_rats = 3: 1.5444498726113796

test RMSE of a tuned bias-only model: 1.5818419897041982
test NDCG of a tuned bias-only model: 0.00026653465444850984


## Full model - tuning

In [23]:
# k, l_bias, l_fact
to_try = [
    [384, 3.3, 10.5],
    [384, 3.5, 10.5],
]

In [24]:
for k, l_b, l_f in to_try:
    gc.collect()
    gc.collect()
    print(f"{k = :>3}, {l_b = :>5.2f}, {l_f = :>5.2f}", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V = fit_model_full(
        Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, valid_data, 1e-5, info=1
    )

k = 384, l_b =  3.30, l_f = 10.50, results: counter =  11, max_norm_diff =    1.62, valid_rmse = 1.597300, valid_ndcg = 0.000533, valid_rmse_diff = 4.5e-06, valid_ndcg_diff = 0      
k = 384, l_b =  3.50, l_f = 10.50, results: counter =  11, max_norm_diff =    1.62, valid_rmse = 1.597372, valid_ndcg = 0.000533, valid_rmse_diff = 5.6e-06, valid_ndcg_diff = 0      


## Full model - testing

In [26]:
# k, l_bias, l_fact
to_try = [
    [1, 4.4, 24.5],
    [2, 4.4, 20],
    [4, 4.4, 21.5],
    [8, 4.4, 21.5],
    [16, 4.3, 14.7],
    [32, 4.3, 15],
    [64, 4.3, 15],
    [128, 4.2, 12.5],
    [256, 4.1, 10.5],
    [384, 4.1, 10.5],
]

In [ ]:
print("test rmse of a tuned model with only biases:", test_rmse_biases)

for k, l_b, l_f in to_try:
    print(f"{k = :>3}, {l_b = :>4.2f}, {l_f = :>4.2f}", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V, valid_rmse = fit_model_full(
        Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, valid_data, 1e-5, info=1
    )
    biases = mu + b[test_data.rows] + c[test_data.cols]
    preds = np.clip(
        biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10
    )
    print(f"     test rmse of a full model with {k = :>3}:", get_rmse(preds, test_data))